# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ishwarsolanki-004/ML-internship-with-flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data Contract

1. **Unit of analysis:** One source row represents one content page for one client on one report date (`report_date × client_hash_id × content_hash_id`).

2. **Table:** I will use `fact_content_daily_performance` because it contains daily search and analytics performance signals for content pages.

3. **Time window:** I will use March 2026 (`2026-03-01` to `2026-03-31`) as my mid-panel development month. June 2026 will remain a sealed final test month.

4. **Output / proxy:** My lane is Content Refresh / Opportunity Scoring. The intended output is a ranked list of content pages for editorial review using observed search and engagement signals.

5. **Deliberate exclusion:** I will not use identifiers or future/label-derived information as model features. IDs are only for grouping and joining, while future or label-derived information would create leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification

**Features**
- `gsc_impressions` — observed search visibility.
- `gsc_clicks` — observed search clicks.
- `gsc_avg_position` — observed average search ranking position.
- `ga4_pageviews` — observed page traffic.
- `ga4_engaged_sessions` — observed engagement.

I will use a maximum of five features and aggregate them over the March 2026 development window.

**Label / Proxy**
- The project output is a refresh-opportunity ranking for editorial review.
- Any field used to define the evaluation label/proxy will not also be used as an honest model feature.

**Context**
- `report_date` — defines the observation window.
- `client_hash_id` — used for grouping or splitting, not as a model feature.
- `content_hash_id` — identifies the content item and is used for grouping, not as a model feature.
- `gsc_data_available` — indicates whether GSC measurements are available.
- `ga4_data_available` — indicates whether GA4 measurements are available.

**Excluded**
- `client_hash_id` and `content_hash_id` are excluded from model features because they are pseudonymous identifiers rather than performance signals.
- Rows where the required source is unavailable will not be interpreted as genuine zero activity.
- Future or label-derived fields are excluded from the honest feature set because they would leak information about the outcome.
- June 2026 is excluded from development because it is the final panel month and is reserved as a sealed test window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification

I use the March 2026 partition as the development slice and verify three facts before building features:

1. **Grain:** each `report_date × client_hash_id × content_hash_id` combination should occur at most once.
2. **Count and date span:** measure the number of rows in the March slice and confirm its observed date range.
3. **Availability:** measure how many rows have both GSC and GA4 data available using `IS TRUE`.

In [1]:
import os
import duckdb
from huggingface_hub import hf_hub_download

# Read Hugging Face token from environment
token = os.getenv("HF_TOKEN")

if token is None:
    raise ValueError("HF_TOKEN was not found in the environment.")

# Download/cache the March 2026 development partition
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token
)

con = duckdb.connect()

print("March 2026 development data ready.")

March 2026 development data ready.


In [2]:
import duckdb

con = duckdb.connect()

grain_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(?)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(
    grain_query,
    [file_path]
).df()

print("Duplicate grain combinations:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations: 0


,report_date,client_hash_id,content_hash_id,row_count


In [3]:
# Query 2: Row count and observed date span

count_window_query = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(?)
"""

count_window = con.execute(
    count_window_query,
    [file_path]
).df()

count_window

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [4]:
# Query 3: Check usable rows with both GSC and GA4 available

availability_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS rows_with_both_available
FROM read_parquet(?)
"""

availability_check = con.execute(
    availability_query,
    [file_path]
).df()

availability_check

,total_rows,rows_with_both_available
0,9841378,364347


### Five-feature frame

I use only rows where both GSC and GA4 data are available and aggregate the March daily observations to the content-page level.

The five features are:

1. **gsc_impressions_30d** — knowable at the decision moment because it summarizes search impressions already observed during the completed March window.
2. **gsc_clicks_30d** — knowable at the decision moment because it uses search clicks already observed during the completed March window.
3. **gsc_avg_position_30d** — knowable at the decision moment because it summarizes search ranking positions observed during the completed March window.
4. **ga4_pageviews_30d** — knowable at the decision moment because it uses pageviews already measured during the completed March window.
5. **ga4_engaged_sessions_30d** — knowable at the decision moment because it uses engagement already measured during the completed March window.

`client_hash_id` and `content_hash_id` remain context fields for grouping and are not model features.

In [5]:
feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions_30d,
    SUM(gsc_clicks) AS gsc_clicks_30d,
    AVG(gsc_avg_position) AS gsc_avg_position_30d,
    SUM(ga4_pageviews) AS ga4_pageviews_30d,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_30d
FROM read_parquet(?)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

feature_frame = con.execute(
    feature_query,
    [file_path]
).df()

print("Content-page feature rows:", len(feature_frame))
feature_frame.head()

Content-page feature rows: 63856


,client_hash_id,content_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,ga4_pageviews_30d,ga4_engaged_sessions_30d
0,client_e547b89c05043229,content_e2c23c916c67ad13,354.0,2.0,13.407452,4.0,0.0
1,client_e547b89c05043229,content_88ff7b9524b78a6f,5786.0,45.0,5.037215,46.0,5.0
2,client_e547b89c05043229,content_2029e7af04068f6f,2481.0,9.0,19.582060,14.0,2.0
3,client_e547b89c05043229,content_241ab01426122c1b,325.0,3.0,24.167785,6.0,1.0
4,client_e547b89c05043229,content_f478e12d031c2cf6,5147.0,5.0,3.065987,13.0,0.0


### Deliberate Leakage Experiment

For this leakage demonstration, I split March 2026 into two time windows:

- **Observation window:** March 1–21, used to build features available at the decision moment.
- **Outcome window:** March 22–31, used only to define a directional proxy label.

The proxy label is whether a page's average daily GSC impressions declined in the outcome window compared with the observation window.

I first train a quick model using only information available from the observation window. I then deliberately add one column derived from the outcome window. This future-derived column leaks information about the label and should make the evaluation score unrealistically strong.

Finally, I remove the leaked column and retain the honest score.

This proxy is used only to demonstrate leakage; it is not claimed to be the final refresh-opportunity label.

In [6]:
leakage_query = """
WITH page_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Observation window: March 1-21
        SUM(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        ) AS impressions_obs,

        SUM(gsc_clicks) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        ) AS clicks_obs,

        AVG(gsc_avg_position) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        ) AS avg_position_obs,

        SUM(ga4_pageviews) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        ) AS pageviews_obs,

        SUM(ga4_engaged_sessions) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        ) AS engaged_sessions_obs,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        ) AS obs_days,

        -- Future/outcome information
        SUM(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
        ) AS impressions_future,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
        ) AS future_days

    FROM read_parquet(?)

    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_windows
WHERE obs_days > 0
  AND future_days > 0
"""

leakage_df = con.execute(
    leakage_query,
    [file_path]
).df()

print("Pages available in both windows:", len(leakage_df))
leakage_df.head()

Pages available in both windows: 28530


,client_hash_id,content_hash_id,impressions_obs,clicks_obs,avg_position_obs,pageviews_obs,engaged_sessions_obs,obs_days,impressions_future,future_days
0,client_e547b89c05043229,content_30d414c2defff507,17921.0,91.0,18.491996,46.0,2.0,15,5463.0,5
1,client_e547b89c05043229,content_8d7d99f109e19aa2,75112.0,251.0,2.624592,138.0,15.0,19,106830.0,9
2,client_e547b89c05043229,content_f59c8de7c55b528a,235.0,2.0,6.673354,4.0,0.0,4,218.0,2
3,client_e547b89c05043229,content_7fd5645147715eba,3458.0,32.0,3.945130,37.0,2.0,14,2140.0,8
4,client_e547b89c05043229,content_67b05bd8d278f6c1,2385.0,15.0,3.245388,16.0,1.0,9,1156.0,4


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_df = leakage_df.copy()

# Average daily impressions in each window
model_df["obs_avg_impressions"] = (
    model_df["impressions_obs"] / model_df["obs_days"]
)

model_df["future_avg_impressions"] = (
    model_df["impressions_future"] / model_df["future_days"]
)

# Proxy label:
# 1 = average daily impressions declined in the future window
# 0 = did not decline
model_df["declined_label"] = (
    model_df["future_avg_impressions"]
    < model_df["obs_avg_impressions"]
).astype(int)

print("Total pages:", len(model_df))
print("\nLabel distribution:")
print(model_df["declined_label"].value_counts())

Total pages: 28530

Label distribution:
declined_label
0    15444
1    13086
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Honest features: only information from March 1-21
honest_features = [
    "impressions_obs",
    "clicks_obs",
    "avg_position_obs",
    "pageviews_obs",
    "engaged_sessions_obs"
]

X = model_df[honest_features].fillna(0)
y = model_df["declined_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_probability = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    honest_probability
)

print("Honest ROC-AUC:", round(honest_auc, 4))

Honest ROC-AUC: 0.5806


#### Deliberate leak

The honest model achieved a ROC-AUC of 0.5830 using only observation-window features.

I now deliberately add one label-derived column, `decline_leak`, which directly copies the proxy label. This column would never be available at the decision moment. It is included only to demonstrate how target leakage can create an unrealistically strong evaluation score.

In [9]:
# Deliberate target leakage:
# This column directly reveals the label.

model_df["decline_leak"] = model_df["declined_label"]

leaked_features = honest_features + ["decline_leak"]

X_leaked = model_df[leaked_features].fillna(0)
y = model_df["declined_label"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaked,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

leaked_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaked_model.fit(X_train_leak, y_train_leak)

leaked_probability = leaked_model.predict_proba(X_test_leak)[:, 1]

leaked_auc = roc_auc_score(
    y_test_leak,
    leaked_probability
)

print("Honest ROC-AUC:", round(honest_auc, 4))
print("Leaked ROC-AUC:", round(leaked_auc, 4))

Honest ROC-AUC: 0.5806
Leaked ROC-AUC: 1.0


In [10]:
# Remove the deliberately leaked feature
model_df = model_df.drop(columns=["decline_leak"])

print("Leak removed:", "decline_leak" not in model_df.columns)
print("Final retained honest ROC-AUC:", round(honest_auc, 4))

Leak removed: True
Final retained honest ROC-AUC: 0.5806


#### Leakage lesson

Adding the label-derived `decline_leak` feature increased ROC-AUC from 0.5830 to 1.0000. This was not a real improvement: the feature directly revealed the outcome.

I removed the leaked feature and retained the honest ROC-AUC of 0.5830. This demonstrates why every feature must be checked for whether it was genuinely available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations

A key limitation is uneven data availability across clients and dates. In the March 2026 slice, there are 9,841,378 total daily rows, but only 364,347 rows have both GSC and GA4 data available.

Rows from periods where GA4 data is unavailable cannot be treated as genuine zero engagement because the warehouse may contain zero-filled GA4 values with `ga4_data_available = FALSE`. I therefore filter on the availability flags before using these measurements.

The refresh-opportunity result should therefore be treated as directional decision-support for pages with usable history, not as a complete measurement of every content page.

In [11]:
# Reuse the already-computed availability result; no additional warehouse query.

total_rows = int(availability_check["total_rows"].iloc[0])
usable_rows = int(availability_check["rows_with_both_available"].iloc[0])

usable_pct = (usable_rows / total_rows) * 100

print("Total March rows:", total_rows)
print("Rows with both GSC and GA4 available:", usable_rows)
print("Share with both available:", round(usable_pct, 2), "%")

Total March rows: 9841378
Rows with both GSC and GA4 available: 364347
Share with both available: 3.7 %


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.